# ДЗ №3. Классификация акустических событий (ESC-10)

**Задача:** обучить **собственную небольшую CNN** распознавать 10 типов окружающих звуков по **log-Mel-спектрограммам** и сравнить результаты **двух** экспериментов, отличающихся **ровно одним** фактором.

В ноутбуке **уже готово**: чтение и проверка набора данных, фиксированное разбиение по folds, извлечение/кэширование log-Mel, `Dataset` и `DataLoader`.  

**Ваша работа:** написать модель, цикл train/val, метрики, контрольный эксперимент, визуализацию и выводы в ячейках `TODO`.

**Не используйте** готовые аудиомодели и предобученные веса. **Наличие GPU не обязательно**: используйте небольшую сеть, не меняя установленный состав данных и число эпох. Готовый результат — один `solution.ipynb` с сохранёнными выводами ячеек.

## 1. Подготовка среды и скачивание данных

1. Скачайте полный архив [ESC-50 с официального GitHub](https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip) (около **600 МБ**). Он содержит также нужное нам подмножество **ESC-10** — 400 пятисекундных WAV-записей.
2. Распакуйте архив **в `data/` рядом с `solution.ipynb`**, сохраняя структуру каталогов:

   ```text
   ваша_папка/
   ├── solution.ipynb
   ├── data/
   │   └── ESC-50-master/
   │       ├── audio/           # 2000 файлов .wav
   │       └── meta/
   │           └── esc50.csv    # содержит признак esc10
   ```

   В Windows можно открыть скачанный ZIP в Проводнике и распаковать папку `ESC-50-master` внутрь `data/`. В Linux/macOS: `unzip ESC-50-master.zip -d data` из каталога с ноутбуком.

3. Установите зависимости **один раз** в том же окружении, где запускается Jupyter:

   ```bash
   python -m pip install -r requirements.txt
   ```

   Если файла `requirements.txt` нет, выполните `python -m pip install torch numpy pandas "librosa>=0.10.2" soundfile matplotlib scikit-learn ipython`.

4. Запустите ячейки ниже: код **сам выберет** 400 записей ESC-10 по `esc10 == True`. В первую обработку лог-Mel будет создан кэш в `data/`, чтобы избежать повторных вычислений. При изменении настроек признаки нужно пересоздать, но **в этой работе параметры менять нельзя**.

**Внимание:** не скачивайте отдельную старую версию ESC-10 в другом формате: этот ноутбук рассчитан на структуру официального репозитория ESC-50 с `.wav` и `meta/esc50.csv`.

## 2. Импорты и фиксированные настройки

In [ ]:
from pathlib import Path
import random

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
SAMPLE_RATE = 16_000
DURATION_S = 5
N_MELS = 64
N_FFT = 1024
HOP_LENGTH = 256
BATCH_SIZE = 16
MIN_EPOCHS = 8
TRAIN_FOLDS = (1, 2, 3, 4)
VAL_FOLD = 5
NUM_CLASSES = 10
NUM_WORKERS = 0  # Надёжно работает и в Jupyter на Windows
DATA_DIR = Path('data')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_random_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Для гарантированной bitwise-воспроизводимости на GPU могут потребоваться
    # дополнительные настройки backend / deterministic algorithms.


set_random_seeds(SEED)
print('PyTorch:', torch.__version__, '| librosa:', librosa.__version__, '| device:', DEVICE)

## 3. Метаданные и фиксированное разбиение

Ищем `meta/esc50.csv` внутри `data/`, выбираем строки с `esc10 == True`, строим **собственную последовательную нумерацию меток 0…9**. Нельзя использовать исходный столбец `target` как метку задачи ESC-10: в полном ESC-50 его значения находятся в диапазоне 0…49.

**Обязательные размеры:** всего 400 записей, 10 классов, по 40 записей на класс. Обучение — folds 1–4 (320 файлов, 32 на класс); валидация — fold 5 (80 файлов, 8 на класс). Состав данных **не менять**.

In [ ]:
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f'Нет {DATA_DIR.resolve()}. Скачайте ESC-50 и распакуйте папку ESC-50-master в data/.'
    )

candidates = sorted(DATA_DIR.rglob('esc50.csv'))
candidates = [p for p in candidates if p.parent.name == 'meta' and (p.parent.parent / 'audio').is_dir()]
if not candidates:
    raise FileNotFoundError(
        'В data/ не найдены meta/esc50.csv и соседняя папка audio/. '
        'Проверьте структуру полного архива ESC-50.'
    )
if len(candidates) > 1:
    raise RuntimeError(f'Найдено несколько копий датасета: {candidates}. Оставьте одну.')

META_CSV = candidates[0]
AUDIO_DIR = META_CSV.parent.parent / 'audio'
meta = pd.read_csv(META_CSV)
meta['esc10'] = meta['esc10'].astype(str).str.lower().eq('true')
esc10_df = meta.loc[meta['esc10']].copy()
label_names = sorted(esc10_df['category'].unique().tolist())
label_to_idx = {name: i for i, name in enumerate(label_names)}
esc10_df['label'] = esc10_df['category'].map(label_to_idx).astype(int)
esc10_df = esc10_df.sort_values(['fold', 'filename']).reset_index(drop=True)

assert len(esc10_df) == 400, f'Ожидается 400 записей, получено {len(esc10_df)}'
assert len(label_names) == NUM_CLASSES
assert esc10_df.groupby('category').size().eq(40).all()
assert set(esc10_df['fold'].unique()) == {1, 2, 3, 4, 5}
assert esc10_df.groupby(['fold', 'category']).size().eq(8).all()

missing_files = [name for name in esc10_df['filename'] if not (AUDIO_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f'Не найдены {len(missing_files)} WAV-файлов ESC-10. Первый: {missing_files[0]}. '
        'Нужно распаковать полный архив, а не только метаданные.'
    )

train_mask = esc10_df['fold'].isin(TRAIN_FOLDS).to_numpy()
val_mask = esc10_df['fold'].eq(VAL_FOLD).to_numpy()
train_idx = np.flatnonzero(train_mask)
val_idx = np.flatnonzero(val_mask)
assert len(train_idx) == 320 and len(val_idx) == 80
assert set(train_idx).isdisjoint(set(val_idx))

print('Метаданные:', META_CSV)
print('Количество файлов: train =', len(train_idx), '| val =', len(val_idx))
print('Классы (индекс → имя):')
print(pd.DataFrame({'label': list(range(NUM_CLASSES)), 'class': label_names}).to_string(index=False))
print('\nПроверка классов по folds:')
display(pd.crosstab(esc10_df['category'], esc10_df['fold']))

## 4. Послушаем и рассмотрим записи

Выведите три фрагмента **разных классов**: запись для воспроизведения, её волновую форму и log-Mel-спектрограмму. **TODO для отчёта:** в текстовой ячейке после графиков кратко опишите, чем различаются эти звуки по длительности, ритму, частотам или характеру шума.

In [ ]:
def read_waveform(filename: str) -> np.ndarray:
    """Mono float32 и ресемплинг до заданной частоты."""
    waveform, native_sr = sf.read(AUDIO_DIR / filename, dtype='float32')
    if waveform.ndim == 2:
        waveform = waveform.mean(axis=1)
    if native_sr != SAMPLE_RATE:
        waveform = librosa.resample(waveform, orig_sr=native_sr, target_sr=SAMPLE_RATE)
    expected = int(SAMPLE_RATE * DURATION_S)
    # WAV обычно имеет нужную длину, pad/truncate страхует от единичных отсчётов.
    if len(waveform) < expected:
        waveform = np.pad(waveform, (0, expected - len(waveform)))
    else:
        waveform = waveform[:expected]
    return waveform.astype(np.float32, copy=False)


def waveform_to_logmel(waveform: np.ndarray) -> np.ndarray:
    mel_power = librosa.feature.melspectrogram(
        y=waveform, sr=SAMPLE_RATE, n_fft=N_FFT,
        hop_length=HOP_LENGTH, n_mels=N_MELS, power=2.0,
    )
    logmel = librosa.power_to_db(mel_power, ref=1.0, top_db=80.0)
    return logmel.astype(np.float32)

selected_examples = esc10_df.drop_duplicates('category').iloc[:3]
fig, axes = plt.subplots(3, 2, figsize=(12, 9), constrained_layout=True)
for i, row in enumerate(selected_examples.itertuples()):
    wav = read_waveform(row.filename)
    spec = waveform_to_logmel(wav)
    axes[i, 0].plot(np.arange(len(wav)) / SAMPLE_RATE, wav, linewidth=0.55)
    axes[i, 0].set(title=f'{row.category}: waveform', xlabel='Time, s', ylabel='Amplitude')
    im = axes[i, 1].imshow(spec, origin='lower', aspect='auto', cmap='magma',
                           extent=(0, DURATION_S, 0, N_MELS))
    axes[i, 1].set(title=f'{row.category}: log-Mel (dB)', xlabel='Time, s', ylabel='Mel bin')
    fig.colorbar(im, ax=axes[i, 1], shrink=0.72)
plt.show()
for row in selected_examples.itertuples():
    print('Пример:', row.category, '|', row.filename)
    display(Audio(data=read_waveform(row.filename), rate=SAMPLE_RATE))

**TODO (ваш текст):** сравните три записи по наблюдаемым особенностям волновой формы и спектрограммы.

## 5. Подготовка и кэширование log-Mel-признаков

Все записи преобразуем одинаково: 5 секунд при 16 кГц → **64 Mel-канала**, `n_fft=1024`, `hop_length=256`. На диске будет создан небольшой кэш с признаками. Его **можно удалить** — ноутбук пересчитает признаки. При двух запусках CNN кэш позволяет не декодировать WAV повторно.

In [ ]:
CACHE_FILE = DATA_DIR / 'esc10_logmel_sr16000_mel64_fft1024_hop256_v1.npz'
expected_filenames = esc10_df['filename'].to_numpy(dtype=str)

if CACHE_FILE.is_file():
    with np.load(CACHE_FILE) as cached:
        cached_names = cached['filenames'].astype(str)
        if not np.array_equal(cached_names, expected_filenames):
            raise RuntimeError('Кэш создан для другого состава/порядка файлов. Удалите ' + str(CACHE_FILE))
        X_logmel = cached['features'].astype(np.float32)
    print('Прочитан кэш:', CACHE_FILE)
else:
    features = []
    print(f'Подготовка {len(esc10_df)} записей. Выполняется только при первом запуске…')
    for n, filename in enumerate(expected_filenames, start=1):
        features.append(waveform_to_logmel(read_waveform(filename)))
        if n % 100 == 0:
            print(f'Обработано {n}/{len(expected_filenames)}')
    X_logmel = np.stack(features, axis=0).astype(np.float32)
    np.savez_compressed(CACHE_FILE, features=X_logmel, filenames=expected_filenames)
    print('Сохранён кэш:', CACHE_FILE)

expected_frames = 1 + (SAMPLE_RATE * DURATION_S) // HOP_LENGTH
assert X_logmel.shape == (400, N_MELS, expected_frames), X_logmel.shape
assert np.isfinite(X_logmel).all(), 'В спектрограммах обнаружены NaN/Inf'
print('Форма признаков [N, Mel, T]:', X_logmel.shape)

## 6. Нормализация, `Dataset` и `DataLoader`

Среднее и стандартное отклонение считаем **только на обучении** (folds 1–4) и применяем эти же числа к валидации. Иначе произошла бы небольшая **утечка информации из val**. При каждом новом эксперименте используйте `make_dataloaders(SEED)` для одинаковой перестановки train-примеров.

In [ ]:
train_mean = float(X_logmel[train_idx].mean())
train_std = float(X_logmel[train_idx].std())
assert train_std > 1e-6
X_normalized = ((X_logmel - train_mean) / train_std)[:, np.newaxis, :, :].astype(np.float32)
y_all = esc10_df['label'].to_numpy(dtype=np.int64)


class ESC10Dataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        global_index = int(self.indices[index])
        spectrogram = torch.from_numpy(X_normalized[global_index])  # [1, 64, T], float32
        label = torch.tensor(y_all[global_index], dtype=torch.long)
        return spectrogram, label


train_dataset = ESC10Dataset(train_idx)
val_dataset = ESC10Dataset(val_idx)


def make_dataloaders(seed=SEED):
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        generator=torch.Generator().manual_seed(seed),
        num_workers=NUM_WORKERS,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    )
    return train_loader, val_loader


train_loader, val_loader = make_dataloaders(SEED)
inputs, targets = next(iter(train_loader))
print('Batch inputs:', tuple(inputs.shape), inputs.dtype)
print('Batch targets:', tuple(targets.shape), targets.dtype)
print('Normalization (train only): mean=', round(train_mean, 3), 'std=', round(train_std, 3))
assert inputs.shape == (BATCH_SIZE, 1, N_MELS, expected_frames)
assert targets.dtype == torch.long and targets.min() >= 0 and targets.max() < NUM_CLASSES
assert len(train_dataset) == 320 and len(val_dataset) == 80

## 7. Самостоятельная реализация CNN — `TODO`

Напишите **свою компактную сеть**, используя только базовые слои PyTorch. Можно использовать `Conv2d`, `ReLU`, `MaxPool2d`, `AdaptiveAvgPool2d`, `Flatten`, `Linear`, `Dropout`, `BatchNorm2d`. Это примеры допустимых компонентов.

**Формат входных и выходных данных:** вход `[B, 1, 64, 313]` -> **логиты** `[B, 10]` (без `softmax`). Для обучения на CPU начинайте с небольшого числа каналов и глобального усреднения в конце.

**TODO:** напишите архитектуру и в следующей текстовой ячейке кратко объясните, как меняется форма тензора.

In [ ]:
class AudioCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        # TODO: создайте собственные свёрточные блоки и классификационную голову.
        raise NotImplementedError('Реализуйте архитектуру AudioCNN')

    def forward(self, x):
        # TODO: верните логиты [batch, num_classes].
        raise NotImplementedError('Реализуйте прямой проход')


def check_model_output_shape(model: nn.Module, input_batch: torch.Tensor):
    model = model.to(DEVICE).eval()
    with torch.no_grad():
        logits = model(input_batch.to(DEVICE))
    expected = (len(input_batch), NUM_CLASSES)
    assert tuple(logits.shape) == expected, (tuple(logits.shape), expected)
    assert torch.isfinite(logits).all()
    print('Проверка формы модели пройдена:', tuple(logits.shape))

# После реализации AudioCNN раскомментируйте строки:
# set_random_seeds(SEED)
# model = AudioCNN()
# check_model_output_shape(model, inputs)

**TODO (ваш текст):** слои и размеры активаций в вашей CNN, число обучаемых параметров, почему модель подходит для CPU.

## 8. Метрики классификации — `TODO`

Собирайте предсказания модели для **всех 80** примеров из `val`, а не вычисляйте среднее значение метрики по батчам». Зафиксируйте:

- `val accuracy` — доля верных классов;
- `val macro-F1` — среднее F1 по 10 классам с равными весами;
- `val loss` — средняя потеря по примерам;
- `confusion_matrix` — матрица ошибок с подписанными классами.

Используйте `sklearn.metrics` с `labels=list(range(NUM_CLASSES))` и `zero_division=0` для расчёта macro-F1. Метрики рассчитываются по предсказанным меткам `argmax(logits, dim=1)`.

`CrossEntropyLoss` принимает именно **логиты** `[B, 10]` и метки `torch.long` `[B]`. Вызывать `softmax` перед ней **не нужно**.

In [ ]:
# TODO: добавьте расчёт accuracy и macro-F1 по накопленным y_true/y_pred.
# Пример корректных аргументов (после того, как вы собрали массивы):
# val_acc = accuracy_score(y_true, y_pred)
# val_f1 = f1_score(y_true, y_pred, labels=list(range(NUM_CLASSES)),
#                   average='macro', zero_division=0)
# cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
# Подписывайте оси матрицы ошибок элементами label_names.

## 9. Обучение базовой CNN — `TODO`

1. Вызовите `set_random_seeds(SEED)` **до** создания модели и оптимизатора.
2. Вызовите `make_dataloaders(SEED)`, обучайте **не менее 8 эпох**.
3. На train обновляйте веса: `optimizer.zero_grad()` → `loss.backward()` → `optimizer.step()`.
4. На val используйте `model.eval()` и `with torch.no_grad():`, не обновляйте веса.
5. Сохраняйте `train_loss`, `val_loss`, `val_accuracy`, `val_macro_f1` **после каждой эпохи**.

При вычислении средней потери по эпохе учитывайте размеры батчей: суммируйте `loss.item() * batch_size`, затем делите на количество примеров, а не на число батчей (если размер последнего батча отличается).

In [ ]:
# TODO: создайте модель, функцию потерь, оптимизатор и train/val циклы.
# Пример ожидаемой структуры history_baseline:
# history_baseline = {
#     'train_loss': [...],
#     'val_loss': [...],
#     'val_accuracy': [...],
#     'val_macro_f1': [...],
# }
# Сохраните финальные предсказания и классы по всем 80 val-записям.
# Сохраните число эпох, LR, optimizer и краткое описание архитектуры.

## 10. Контролируемый эксперимент — `TODO`

Сформулируйте гипотезу **до запуска**. Измените **одну** настройку: например, только learning rate, только вероятность dropout либо только аугментацию *обучающих* спектрограмм. Состав `train/val` и базовая архитектура должны оставаться прежними.

Снова выполните `set_random_seeds(SEED)`, создайте **новую** модель и новые загрузчики. Для обоих запусков используйте не менее **8 эпох** и сравнимые вычислительные ресурсы. Изменение итоговой метрики может быть любым — важно корректное сравнение.

In [ ]:
# TODO: проведите второй полный запуск и сохраните history_experiment.
# set_random_seeds(SEED)
# train_loader_experiment, val_loader_experiment = make_dataloaders(SEED)
# model_experiment = AudioCNN(...)  # та же архитектура и seed
# ... измените ровно один фактор и обучите >= 8 эпох

## 11. Сравнение, матрица ошибок и анализ — `TODO`

Представьте в конце ноутбука:

1. Графики `train_loss`, `val_loss`, `val_accuracy`, `val_macro_f1` по эпохам **для обоих запусков**.
2. Матрицу ошибок базовой модели на **всех 80** записях из `val`, с подписями классов.
3. Не менее **трёх** валидационных звуков с истинным и предсказанным классами. Для каждого можно воспроизвести аудио и посмотреть спектрограмму. Разберите минимум **одну** ошибку; если ошибок нет, выберите случай с минимальной уверенностью модели.
4. Сравнительную таблицу по образцу ниже и короткий вывод по гипотезе.

| Запуск | Изменённый фактор | Эпох | LR | Train loss | Val loss | Val accuracy | Val macro-F1 |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Базовый | — | | | | | | |
| Контрольный | | | | | | | |

Не делайте вывод только по одному числу. Проанализируйте графики, матрицу ошибок, особенности акустических классов и ограниченность датасета (по 8 записей каждого класса в val).

In [ ]:
# TODO: построить графики двух запусков, confusion matrix и таблицу.
# Для воспроизведения записи из val можно использовать:
# row = esc10_df.iloc[int(val_idx[0])]
# display(Audio(data=read_waveform(row['filename']), rate=SAMPLE_RATE))
# Печатайте true/pred и при желании вероятность/уверенность ответа.

## 12. Итоговые выводы — `TODO`

Ответьте кратко и по существу:

- Удалось ли обучить базовую модель? Какие факты это подтверждают?
- Какие классы классифицируются хорошо, какие путаются, и чем это можно объяснить?
- Что изменилось во втором эксперименте? Подтвердилась ли гипотеза?
- Каковы ограничения результатов на 400 коротких записях?

**Перед сдачей:** сохраните ноутбук **с уже выполненными ячейками, результатами и графиками**. Не прикладывайте скачанные WAV, кэш `data/*.npz` и большие веса модели. Сдаётся только `solution.ipynb`.